# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MuhammadOmerSiddiqui/myInternship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [5]:
import os, sys, subprocess
import numpy as np
import pandas as pd

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score, average_precision_score

# Repo root
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    REPO = "myInternship"
    if not os.path.isdir(REPO):
        subprocess.run(
            ["git", "clone", "--depth", "1",
             "https://github.com/MuhammadOmerSiddiqui/myInternship.git", REPO],
            check=True
        )
    os.chdir(REPO)

csv_path = "data/raw/content_refresh_anonymized.csv"
assert os.path.exists(csv_path), f"CSV not found. cwd={os.getcwd()}"

df = pd.read_csv(csv_path)
df["is_declining"] = (df["trend_direction"] == "down").astype(int)
print("Loaded", len(df), "pages | declining rate:", round(df["is_declining"].mean(), 3))

Loaded 30000 pages | declining rate: 0.542


## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

**Lane:** Refresh / Content Opportunity Scoring  
**Question shape:** “Which pages should an editor review first?” → ranking problem built on a yes/no proxy label.

**Methods I use (from the toolkit):**
1. **Logistic Regression** — simple, readable, good baseline for linear patterns.
2. **Decision Tree (max_depth=5)** — still readable, can capture simple interactions.
3. **Random Forest** — stronger ranking scores when many weak signals interact.

I start simple and only keep the method that clearly beats the Week-4 hand-written rule on the **same metric (Precision@50)** and the **same client-holdout split**.

I do **not** use Gradient Boosting here — the starter data is small enough that RF already shows the point, and complexity alone is not rewarded.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

**Split: client-holdout (grouped by client_id)**

- ~80% of clients → train  
- ~20% of clients → test  
- No page from a test client ever appears in training.

**Why this is honest for my question**
- Different clients have different content styles and traffic patterns.
- A random row split would leak client-specific patterns into both sides and make the model look better than it is.
- Client-holdout answers: “Does this ranking still work on clients the model has never seen?”

Random seed is fixed at 42 so the table is reproducible.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)

clients = df["client_id"].fillna("unknown").astype(str)
unique_clients = clients.drop_duplicates().to_numpy()
shuffled = rng.permutation(unique_clients)
n_test = max(1, int(round(len(shuffled) * 0.20)))
test_clients = set(shuffled[:n_test])

test_mask = clients.isin(test_clients).to_numpy()
train_idx = np.where(~test_mask)[0]
test_idx  = np.where(test_mask)[0]

print("Split strategy : client_holdout")
print("Train rows     :", len(train_idx))
print("Test rows      :", len(test_idx))
print("Train clients  :", len(set(clients.iloc[train_idx])))
print("Test clients   :", len(set(clients.iloc[test_idx])))
print("Test declining rate:", round(df.iloc[test_idx]["is_declining"].mean(), 3))

Split strategy : client_holdout
Train rows     : 27675
Test rows      : 2325
Train clients  : 26
Test clients   : 6
Test declining rate: 0.391


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

**Same data, same split, same metric as Week 4.**

Primary metric: **Precision@50**  
(of the 50 pages ranked highest, how many are actually declining?)

I also report ROC-AUC and Average Precision for context, plus the base rate so the numbers have meaning.

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ---------- features (honest only) ----------
# NEVER use trend_direction / trend_pct / is_declining as features
numeric_cols = [
    "impressions_90d", "clicks_90d", "sessions_90d",
    "content_age_days", "days_since_last_update",
    "avg_position", "ctr", "engagement_rate", "scroll_rate",
    "word_count", "char_count",
    "days_with_impressions", "days_with_sessions",
    "ai_sessions_90d", "ai_traffic_pct",
]
cat_cols = ["content_type", "main_intent", "competition_level",
            "age_tier", "freshness_tier", "impression_tier", "position_tier"]

# keep only columns that exist
numeric_cols = [c for c in numeric_cols if c in df.columns]
cat_cols     = [c for c in cat_cols if c in df.columns]

X_num = df[numeric_cols].apply(pd.to_numeric, errors="coerce").replace([np.inf, -np.inf], np.nan).fillna(0)
X_cat = pd.get_dummies(df[cat_cols].fillna("unknown").astype(str), dtype=float)
X = pd.concat([X_num.reset_index(drop=True), X_cat.reset_index(drop=True)], axis=1)
y = df["is_declining"].astype(int)

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

# ---------- rebuild Week-4 style baseline score on the same data ----------
def percentile_rank(s):
    return s.rank(pct=True, method="average").fillna(0)

def normalize(s):
    s = s.astype(float)
    return (s - s.min()) / (s.max() - s.min() + 1e-9)

vis   = percentile_rank(np.log1p(df["impressions_90d"]))
fresh = percentile_rank(df["days_since_last_update"].fillna(0))
pos   = (1 - normalize(df["avg_position"].clip(1, 50))) * vis * (df["avg_position"] > 0).astype(int)
depth = (1 - percentile_rank(df["word_count"].fillna(0))) * vis
baseline_score = (0.40*vis + 0.30*fresh + 0.25*pos + 0.05*depth).clip(0, 1)
baseline_test  = baseline_score.iloc[test_idx].to_numpy()

# ---------- helpers ----------
def precision_at_k(y_true, scores, k=50):
    order = np.argsort(-np.asarray(scores))
    top = np.asarray(y_true)[order[:k]]
    return float(top.mean()) if len(top) else 0.0

def metrics(y_true, scores, name):
    return {
        "model": name,
        "precision@50": round(precision_at_k(y_true, scores, 50), 3),
        "precision@20": round(precision_at_k(y_true, scores, 20), 3),
        "roc_auc": round(roc_auc_score(y_true, scores), 3),
        "avg_precision": round(average_precision_score(y_true, scores), 3),
    }

# ---------- models ----------
models = {
    "logistic_regression": Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(class_weight="balanced", max_iter=1000, random_state=RANDOM_STATE)),
    ]),
    "decision_tree": DecisionTreeClassifier(
        class_weight="balanced", max_depth=5, min_samples_leaf=50, random_state=RANDOM_STATE
    ),
    "random_forest": RandomForestClassifier(
        class_weight="balanced_subsample", n_estimators=200, max_depth=10,
        min_samples_leaf=25, n_jobs=-1, random_state=RANDOM_STATE
    ),
}

results = [metrics(y_test, baseline_test, "baseline_rule")]
fitted = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    proba = model.predict_proba(X_test)[:, 1]
    results.append(metrics(y_test, proba, name))
    fitted[name] = (model, proba)

results_df = pd.DataFrame(results).sort_values("precision@50", ascending=False)
print("Base rate (test declining rate):", round(y_test.mean(), 3))
print("\n=== Model vs Baseline (same client-holdout split) ===")
display(results_df)

best_name = results_df.iloc[0]["model"]
print(f"\nBest on Precision@50: {best_name}")

Base rate (test declining rate): 0.391

=== Model vs Baseline (same client-holdout split) ===


,model,precision@50,precision@20,roc_auc,avg_precision
3,random_forest,0.82,0.85,0.749,0.620
1,logistic_regression,0.80,0.80,0.737,0.630
2,decision_tree,0.56,0.50,0.742,0.575
0,baseline_rule,0.24,0.15,0.627,0.468



Best on Precision@50: random_forest


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

**What the model leans on**  
Top features from the best model (Random Forest in most runs): impressions volume, position, content age, days with impressions, CTR. These are the same families of signals the hand-written rule used, but the model can combine them non-linearly.

**Where it is still wrong**
- High-volume stable pages can still rank high (volume dominates).
- Seasonal or consolidated pages look “declining” in the proxy label even when the business decision is “do nothing”.
- Very low-volume pages are noisy; Precision@50 is more reliable than metrics that look at the whole list.

**Takeaway**  
The learned ranking beats the fixed rule on the same split and metric. That is the only claim I need: a better ordered review queue for a human editor, not a guarantee that refreshing will recover traffic.

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Feature importance from the best tree/forest model
best_model_name = results_df[results_df["model"] != "baseline_rule"].iloc[0]["model"]
model_obj, test_proba = fitted[best_model_name]

if hasattr(model_obj, "feature_importances_"):
    imp = pd.Series(model_obj.feature_importances_, index=X.columns).sort_values(ascending=False)
elif hasattr(model_obj, "named_steps"):
    coefs = np.abs(model_obj.named_steps["clf"].coef_[0])
    imp = pd.Series(coefs, index=X.columns).sort_values(ascending=False)
else:
    imp = pd.Series(dtype=float)

print(f"=== Top 10 features ({best_model_name}) ===")
display(imp.head(10).to_frame("importance"))

# Error slice: top-50 predictions vs actual
test_frame = df.iloc[test_idx].copy()
test_frame["model_score"] = test_proba
test_frame["baseline_score"] = baseline_test
top50 = test_frame.nlargest(50, "model_score")

print("\n=== Top-50 of best model ===")
print("Declining in top-50:", int(top50["is_declining"].sum()), "/ 50")
print("Non-declining (false positives) in top-50:", int((1 - top50["is_declining"]).sum()))

# A few concrete false positives
fp = top50[top50["is_declining"] == 0][
    ["content_id", "impressions_90d", "avg_position", "ctr",
     "content_age_days", "trend_direction", "model_score"]
].head(5)
print("\nExample false positives (high score but not declining):")
display(fp)

print("""
Interpretation:
- False positives are often high-impression pages that are stable/up.
- The model still ranks them high because volume + age signals are strong.
- A human editor would still need context (seasonality, consolidation) before acting.
- This is expected for decision-support, not automatic publishing.
""")

=== Top 10 features (random_forest) ===


,importance
days_with_impressions,0.152420
impressions_90d,0.126260
avg_position,0.119697
content_age_days,0.090008
char_count,0.046953
word_count,0.046721
age_tier_365+,0.039058
clicks_90d,0.038662
scroll_rate,0.036433
ctr,0.035112



=== Top-50 of best model ===
Declining in top-50: 41 / 50
Non-declining (false positives) in top-50: 9

Example false positives (high score but not declining):


,content_id,impressions_90d,avg_position,ctr,content_age_days,trend_direction,model_score
4249,content_db1cd41b4b4f,1482,12.9,0.00,105,up,0.770469
23750,content_e55b8ab078b0,369,21.8,0.00,112,stable,0.744038
5966,content_f5013794ba57,881,15.7,0.00,175,new,0.739428
19812,content_ac140b295c0f,1012,26.0,0.00,175,stable,0.736463
23559,content_00603b0349b4,1076,25.6,0.09,125,up,0.734915



Interpretation:
- False positives are often high-impression pages that are stable/up.
- The model still ranks them high because volume + age signals are strong.
- A human editor would still need context (seasonality, consolidation) before acting.
- This is expected for decision-support, not automatic publishing.



## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.